In [ ]:

# Install Required Packages
!pip install requests beautifulsoup4 pandas


In [ ]:

# Step 1: Import Libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# Step 2: Define Constants
BASE_URL = "https://www.bjjheroes.com"
FIGHTER_LIST_URL = "https://www.bjjheroes.com/a-z-bjj-fighters-list"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
DEBUG_MODE = True  # Set to True for debugging, False for full run
DEBUG_LIMIT = 10   # Number of athletes to process in debug mode
RETRY_LIMIT = 3   # Number of retry attempts before logging failure
FAILED_FIGHTERS = []


In [ ]:

# Step 3: Extract Fighter ID from URL
def extract_fighter_id(profile_url):
    """Extracts the fighter ID from the profile URL."""
    match = re.search(r'\?p=(\d+)', profile_url)
    return match.group(1) if match else "Unknown"


In [ ]:

# Step 4: Scrape Fighter List
def get_fighter_list():
    """Extracts fighter details (name, profile URL, and ID) from the main fighter list page."""
    response = requests.get(FIGHTER_LIST_URL, headers=HEADERS)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    fighters = []
    table = soup.find("table")
    if not table:
        print("No fighter table found. The website structure may have changed.")
        return pd.DataFrame()

    rows = table.find_all("tr")[1:]  # Skip header row
    for row in rows:
        cols = row.find_all("td")
        link = row.find("a")  # Get the profile link
        
        if len(cols) >= 3 and link:
            profile_url = BASE_URL + link.get("href") if link.get("href") else "No Link"
            fighter_id = extract_fighter_id(profile_url)
            
            fighter_details = {
                "Fighter ID": fighter_id,
                "First Name": cols[0].text.strip(),
                "Last Name": cols[1].text.strip(),
                "Nickname": cols[2].text.strip(),
                "Team": cols[3].text.strip() if len(cols) > 3 else "Unknown",
                "Profile URL": profile_url
            }
            fighters.append(fighter_details)
            
        if DEBUG_MODE and len(fighters) >= DEBUG_LIMIT:
            break
    
    df = pd.DataFrame(fighters)
    df.to_csv("bjj_fighters.csv", index=False)
    print("Fighter list saved to bjj_fighters.csv")
    return df


In [ ]:

# Step 5: Parse Match Row Data
def parse_match_row(cols, competing_athlete, competing_athlete_id):
    """Parses a row from the match table to extract match ID, opponent, event, method, result, weight, stage, and year."""
    match_id = cols[0].text.strip()
    opponent_name = cols[1].text.strip()
    result = cols[2].text.strip()
    method = cols[3].text.strip()
    event_name = cols[4].text.strip()
    weight = cols[5].text.strip()
    stage = cols[6].text.strip()
    year = cols[7].text.strip()
    
    return {
        "Match ID": match_id,
        "Competing Athlete": competing_athlete,
        "Competing Athlete ID": competing_athlete_id,
        "Opponent": opponent_name,
        "Event": event_name,
        "Method": method,
        "Result": result,
        "Weight": weight,
        "Stage": stage,
        "Year": year
    }


In [ ]:

# Step 6: Scrape Fighter Match Data with Retry Logic
def get_fighter_matches(fighter_name, fighter_id, fighter_url):
    """Extracts match records from a fighter's profile page with retry logic and failure logging."""
    attempts = 0
    while attempts < RETRY_LIMIT:
        try:
            response = requests.get(fighter_url, headers=HEADERS)
            if response.status_code != 200:
                raise Exception(f"HTTP {response.status_code} Error")
            
            soup = BeautifulSoup(response.text, 'html.parser')
            matches = []
            table = soup.find("table")
            if table:
                rows = table.find_all("tr")[1:]  # Skip header row
                for row in rows:
                    cols = row.find_all("td")
                    if len(cols) >= 8:
                        match_details = parse_match_row(cols, fighter_name, fighter_id)
                        matches.append(match_details)
            return matches
        except Exception as e:
            attempts += 1
            print(f"Error scraping {fighter_url} (Attempt {attempts}/{RETRY_LIMIT}): {e}")
            time.sleep(2)
    
    print(f"Failed to scrape: {fighter_url}")
    FAILED_FIGHTERS.append(fighter_url)
    return []


In [ ]:

# Step 7: Scrape and Save Match Data
def scrape_bjj_matches():
    """Scrapes match data for all fighters and saves it to a CSV file."""
    fighter_df = get_fighter_list()
    all_matches = []
    
    for i, row in fighter_df.iterrows():
        fighter_name = f"{row['First Name']} {row['Last Name']}"
        fighter_id = row['Fighter ID']
        fighter_url = row['Profile URL']
        print(f"Scraping matches for {fighter_name}: {fighter_url}")
        matches = get_fighter_matches(fighter_name, fighter_id, fighter_url)
        all_matches.extend(matches)
        
        if DEBUG_MODE and i + 1 >= DEBUG_LIMIT:
            break
        
        time.sleep(1)  # Be polite to the server
    
    df = pd.DataFrame(all_matches)
    df.to_csv("bjj_master_matches.csv", index=False)
    print("Master match data saved to bjj_master_matches.csv")
    
    # Save failure log
    if FAILED_FIGHTERS:
        with open("failed_fighters.log", "w") as f:
            for fighter in FAILED_FIGHTERS:
                f.write(fighter + "\n")
        print("Failed fighter URLs saved to failed_fighters.log")


In [ ]:

# Step 8: Run the Scraper
if __name__ == "__main__":
    scrape_bjj_matches()
